# SQL Query Optimization Report
**Dialect**: PostgreSQL  
**Generated**: 2025-03-14  
**Source**: `queries/ecommerce_analytics.sql`

## Query 1: Order Filtering with Correlated Subquery

### Original Query

```sql
SELECT *
FROM orders o
WHERE o.created_at >= '2024-01-01'
  AND o.created_at < '2025-01-01'
  AND o.status != 'cancelled'
  AND (SELECT COUNT(*)
       FROM order_items oi
       WHERE oi.order_id = o.id) > 0
  AND o.customer_id IN (
    SELECT c.id
    FROM customers c
    WHERE EXTRACT(YEAR FROM c.signup_date) >= 2020
      AND c.is_active = true
  )
ORDER BY o.created_at DESC;
```

### Query Explanation

This query retrieves all columns from the `orders` table for the year 2024, excluding cancelled orders.
It further filters to only include orders that:
1. Have at least one item in `order_items` (checked via a correlated subquery)
2. Belong to active customers who signed up in 2020 or later

Results are sorted by creation date, newest first.

### Execution Flow

Step-by-step database processing order:

1. **FROM**: Sequential scan of `orders` table (assume ~1M rows)
2. **WHERE (date + status)**: Filters to 2024 non-cancelled orders (~200K rows)
3. **WHERE (correlated subquery)**: For **each** of the ~200K rows, runs `SELECT COUNT(*) FROM order_items` -- this is ~200K subqueries
4. **WHERE (IN subquery)**: Scans `customers` table for matching IDs; executed once but compared against every remaining row
5. **ORDER BY**: Sorts all qualifying rows by `created_at DESC`
6. **SELECT ***: Returns all columns (no projection)

**Estimated cost**: The correlated subquery dominates -- O(n * m) where n = filtered orders and m = avg items per order.

### Identified Issues

| Severity | Issue | Location | Impact |
|----------|-------|----------|--------|
| CRITICAL | `SELECT *` returns all columns | Line 1 | Unnecessary I/O, prevents index-only scans |
| CRITICAL | Correlated subquery in WHERE | Lines 5-7 | Executes `COUNT(*)` for every row -- O(n*m) |
| WARNING | `IN (SELECT ...)` anti-pattern | Lines 8-13 | Could be replaced with `EXISTS` for better plan |
| WARNING | No `LIMIT` clause | -- | Returns potentially millions of rows |
| INFO | `!= 'cancelled'` vs positive filter | Line 4 | Negation may prevent index usage |

### Optimized Query

```sql
-- Explicit column list instead of SELECT *
SELECT
    o.id,
    o.customer_id,
    o.created_at,
    o.total_amount,
    o.status
FROM orders o
-- EXISTS replaces the correlated COUNT(*) subquery
-- Database can stop at first match instead of counting all items
WHERE EXISTS (
    SELECT 1 FROM order_items oi WHERE oi.order_id = o.id
)
-- EXISTS replaces IN for the customer filter
AND EXISTS (
    SELECT 1 FROM customers c
    WHERE c.id = o.customer_id
      AND c.signup_date >= '2020-01-01'  -- avoid EXTRACT() on indexed column
      AND c.is_active = true
)
AND o.created_at >= '2024-01-01'
AND o.created_at < '2025-01-01'
AND o.status != 'cancelled'
ORDER BY o.created_at DESC
LIMIT 1000;  -- add pagination
```

**Changes**:
- `SELECT *` replaced with explicit columns
- Correlated `COUNT(*)` replaced with `EXISTS` (short-circuits on first match)
- `IN (SELECT ...)` replaced with `EXISTS` (better execution plan)
- `EXTRACT(YEAR FROM signup_date) >= 2020` replaced with `signup_date >= '2020-01-01'` (index-friendly)
- Added `LIMIT 1000` for pagination

---
## Query 2: Product Category Performance with Running Totals

### Original Query

```sql
SELECT
  p.category,
  p.subcategory,
  DATE_TRUNC('month', o.created_at) AS month,
  SUM(oi.quantity * oi.unit_price) AS revenue,
  COUNT(DISTINCT o.customer_id) AS unique_customers,
  (SELECT SUM(oi2.quantity * oi2.unit_price)
   FROM order_items oi2
   JOIN orders o2 ON o2.id = oi2.order_id
   JOIN products p2 ON p2.id = oi2.product_id
   WHERE p2.category = p.category
     AND o2.created_at <= DATE_TRUNC('month', o.created_at) + INTERVAL '1 month'
     AND o2.status != 'cancelled'
  ) AS running_total
FROM products p
JOIN order_items oi ON oi.product_id = p.id
JOIN orders o ON o.id = oi.order_id
WHERE o.status != 'cancelled'
  AND LOWER(p.category) != 'test'
GROUP BY p.category, p.subcategory, DATE_TRUNC('month', o.created_at)
ORDER BY p.category, month;
```

### Query Explanation

This query computes monthly revenue and unique customer counts per product category and subcategory.
It also calculates a cumulative running total of revenue per category using a correlated subquery
that re-scans the entire `order_items` + `orders` + `products` join for every output row.

### Identified Issues

| Severity | Issue | Location | Impact |
|----------|-------|----------|--------|
| CRITICAL | Correlated subquery for `running_total` | Lines 6-12 | Re-scans entire dataset for each group -- O(n^2) |
| WARNING | `LOWER(p.category)` function on column | WHERE clause | Prevents index usage on `category` |
| INFO | Redundant join in subquery | Lines 8-10 | Could use window function instead |

### Optimized Query

```sql
-- Use a CTE for the base aggregation, then apply a window function for running total
WITH monthly_revenue AS (
    SELECT
        p.category,
        p.subcategory,
        DATE_TRUNC('month', o.created_at) AS month,
        SUM(oi.quantity * oi.unit_price) AS revenue,
        COUNT(DISTINCT o.customer_id) AS unique_customers
    FROM products p
    JOIN order_items oi ON oi.product_id = p.id
    JOIN orders o ON o.id = oi.order_id
    WHERE o.status != 'cancelled'
      AND p.category != 'test'  -- direct comparison, no LOWER()
    GROUP BY p.category, p.subcategory, DATE_TRUNC('month', o.created_at)
)
SELECT
    category,
    subcategory,
    month,
    revenue,
    unique_customers,
    -- Window function replaces the correlated subquery
    SUM(revenue) OVER (
        PARTITION BY category
        ORDER BY month
        ROWS UNBOUNDED PRECEDING
    ) AS running_total
FROM monthly_revenue
ORDER BY category, month;
```

**Changes**:
- Correlated subquery replaced with `SUM() OVER (PARTITION BY ... ORDER BY ...)` window function
- `LOWER(p.category) != 'test'` replaced with direct comparison (store normalized data instead)
- CTE separates aggregation from windowing for clarity

---
## Query 3: Customer Lifetime Value

### Original Query

```sql
SELECT
  c.id,
  c.email,
  c.name,
  c.signup_date,
  COALESCE(
    (SELECT SUM(o.total_amount)
     FROM orders o
     WHERE o.customer_id = c.id
       AND o.status != 'cancelled'),
    0
  ) AS lifetime_value,
  COALESCE(
    (SELECT COUNT(*)
     FROM orders o
     WHERE o.customer_id = c.id
       AND o.status != 'cancelled'),
    0
  ) AS total_orders,
  COALESCE(
    (SELECT MAX(o.created_at)
     FROM orders o
     WHERE o.customer_id = c.id),
    c.signup_date
  ) AS last_order_date,
  CASE
    WHEN (SELECT MAX(o.created_at) FROM orders o WHERE o.customer_id = c.id)
         >= NOW() - INTERVAL '30 days' THEN 'Active'
    WHEN (SELECT MAX(o.created_at) FROM orders o WHERE o.customer_id = c.id)
         >= NOW() - INTERVAL '90 days' THEN 'At Risk'
    ELSE 'Churned'
  END AS segment
FROM customers c
WHERE c.is_active = true
ORDER BY lifetime_value DESC;
```

### Identified Issues

| Severity | Issue | Location | Impact |
|----------|-------|----------|--------|
| CRITICAL | 5 correlated subqueries | Multiple SELECT items | Each scans `orders` table separately -- 5N queries total |
| WARNING | Repeated subquery logic | CASE branches | Same `MAX(created_at)` subquery appears 3 times |
| INFO | Missing `LIMIT` | -- | Returns all active customers |

### Optimized Query

```sql
-- Single LEFT JOIN with aggregation replaces all 5 correlated subqueries
SELECT
    c.id,
    c.email,
    c.name,
    c.signup_date,
    COALESCE(agg.lifetime_value, 0) AS lifetime_value,
    COALESCE(agg.total_orders, 0) AS total_orders,
    COALESCE(agg.last_order_date, c.signup_date) AS last_order_date,
    CASE
        WHEN agg.last_order_date >= NOW() - INTERVAL '30 days' THEN 'Active'
        WHEN agg.last_order_date >= NOW() - INTERVAL '90 days' THEN 'At Risk'
        ELSE 'Churned'
    END AS segment
FROM customers c
LEFT JOIN (
    -- Aggregate orders once per customer
    SELECT
        o.customer_id,
        SUM(CASE WHEN o.status != 'cancelled' THEN o.total_amount ELSE 0 END) AS lifetime_value,
        COUNT(CASE WHEN o.status != 'cancelled' THEN 1 END) AS total_orders,
        MAX(o.created_at) AS last_order_date
    FROM orders o
    GROUP BY o.customer_id
) agg ON agg.customer_id = c.id
WHERE c.is_active = true
ORDER BY lifetime_value DESC
LIMIT 1000;
```

**Changes**:
- All 5 correlated subqueries replaced with a single `LEFT JOIN` on a pre-aggregated subquery
- `MAX(created_at)` computed once and reused in `CASE` (was computed 3 times)
- Added `LIMIT 1000` for pagination
- Orders table scanned exactly once instead of 5 times

## PostgreSQL-Specific Tips

### Recommended Indexes

```sql
-- Composite index for order date-range + status filtering
CREATE INDEX idx_orders_created_status ON orders(created_at, status);

-- Covering index for the customer lookup
CREATE INDEX idx_orders_customer_status ON orders(customer_id, status, total_amount, created_at);

-- Index for the order_items foreign key (often missing)
CREATE INDEX idx_order_items_order_id ON order_items(order_id);

-- Partial index for active customers only
CREATE INDEX idx_customers_active ON customers(id) WHERE is_active = true;
```

### Verification

Run `EXPLAIN ANALYZE` on both original and optimized versions to compare:

```sql
EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
SELECT ... -- your query here
```

Look for:
- **Seq Scan** vs **Index Scan** -- indexes being used
- **Nested Loop** vs **Hash Join** -- join strategy improvement
- **SubPlan** nodes -- correlated subqueries (should disappear in optimized version)
- **Buffers: shared hit** vs **shared read** -- cache efficiency

### Additional Recommendations

- Consider a **materialized view** for the CLV query if it runs frequently
- **Partition** the `orders` table by `created_at` (range partitioning by year/quarter) for faster date-range scans
- Use `pg_stat_statements` to identify other slow queries

## Summary

| Severity | Count | Description |
|----------|------:|-------------|
| CRITICAL | 4 | `SELECT *`, correlated subqueries (x3) |
| WARNING | 4 | `IN` anti-pattern, missing `LIMIT`, repeated subquery, `LOWER()` on indexed column |
| INFO | 3 | Negation filter, redundant join, missing `LIMIT` on CLV |

**Key optimizations applied**:
1. Replaced all correlated subqueries with `EXISTS` or `JOIN` + aggregation
2. Used window functions (`SUM() OVER`) for running totals
3. Eliminated function calls on indexed columns
4. Added explicit column lists and `LIMIT` clauses
5. Recommended supporting indexes